In [0]:
import requests

token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
host = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiUrl().get()

In [0]:


for key, value in secrets.items():
    response = requests.post(
        f"{host}/api/2.0/secrets/put",
        headers={"Authorization": f"Bearer {token}"},
        json={
            "scope": "de-portfolio-scope",
            "key": key,
            "string_value": value
        }
    )
    print(f"{key}: {response.status_code}")

In [0]:
storage_account = "theportfoliostorage"
container = "datalake"

client_id = dbutils.secrets.get(scope="de-portfolio-scope", key="sp-client-id")
client_secret = dbutils.secrets.get(scope="de-portfolio-scope", key="sp-client-secret")
tenant_id = dbutils.secrets.get(scope="de-portfolio-scope", key="sp-tenant-id")

spark.conf.set(f"fs.azure.account.auth.type.{storage_account}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{storage_account}.dfs.core.windows.net", 
               "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{storage_account}.dfs.core.windows.net", client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{storage_account}.dfs.core.windows.net", client_secret)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{storage_account}.dfs.core.windows.net", 
               f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")

print("Storage configuration set successfully")

In [0]:
response = requests.post(
    f"{host}/api/2.1/unity-catalog/volumes",
    headers={"Authorization": f"Bearer {token}"},
    json={
        "catalog_name": "healthcare",
        "schema_name": "bronze",
        "name": "raw_files",
        "volume_type": "MANAGED",
        "comment": "Raw source files for healthcare pipeline"
    }
)
print(response.status_code)
print(response.json())

In [0]:
# Define paths
bronze_path = f"abfss://{container}@{storage_account}.dfs.core.windows.net/raw/healthcare/cms_hrrp/year=2024/"

# Read from Unity Catalog volume
df_raw = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("multiLine", "true") \
    .load("/Volumes/healthcare/bronze/raw_files/")

print(f"Rows loaded: {df_raw.count()}")
print(f"Columns: {len(df_raw.columns)}")
df_raw.printSchema()

In [0]:
# Rename columns to remove spaces and special characters
df_raw_clean = df_raw \
    .withColumnRenamed("Facility Name", "facility_name") \
    .withColumnRenamed("Facility ID", "facility_id") \
    .withColumnRenamed("State", "state") \
    .withColumnRenamed("Measure Name", "measure_name") \
    .withColumnRenamed("Number of Discharges", "number_of_discharges") \
    .withColumnRenamed("Footnote", "footnote") \
    .withColumnRenamed("Excess Readmission Ratio", "excess_readmission_ratio") \
    .withColumnRenamed("Predicted Readmission Rate", "predicted_readmission_rate") \
    .withColumnRenamed("Expected Readmission Rate", "expected_readmission_rate") \
    .withColumnRenamed("Number of Readmissions", "number_of_readmissions") \
    .withColumnRenamed("Start Date", "start_date") \
    .withColumnRenamed("End Date", "end_date")

# Add metadata columns - standard Bronze layer practice
from pyspark.sql.functions import current_timestamp, lit

df_raw_clean = df_raw_clean \
    .withColumn("_ingested_at", current_timestamp()) \
    .withColumn("_source_file", lit("Hospital_Readmissions_Reduction_Program.csv")) \
    .withColumn("_source_year", lit(2024))

print("Columns renamed successfully")
df_raw_clean.printSchema()

In [0]:
df_raw_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("healthcare.bronze.cms_hrrp_raw")

print("Bronze table written successfully")
spark.sql("SELECT COUNT(*) as row_count FROM healthcare.bronze.cms_hrrp_raw").show()

In [0]:
# Verify table exists in Unity Catalog
spark.sql("DESCRIBE EXTENDED healthcare.bronze.cms_hrrp_raw").show(truncate=False)

In [0]:
# Quick sanity check on the data
spark.sql("""
    SELECT 
        facility_name,
        state,
        measure_name,
        excess_readmission_ratio,
        number_of_discharges,
        _ingested_at
    FROM healthcare.bronze.cms_hrrp_raw
    LIMIT 5
""").show(truncate=False)


In [0]:
spark.sql("SHOW TABLES IN healthcare.bronze").show(truncate=False)